In [ ]:
import pandas as pd
import sqlite3
from datetime import datetime

In [ ]:
conn = sqlite3.connect("pokemon.db")
cursor = conn.cursor()

In [ ]:
# Leggi il CSV
df = pd.read_csv("stock.csv")

# Scrive la tabella nel database
df.to_sql(
    "stock",
    conn,
    if_exists="replace",   # "append" se vuoi aggiungere senza sovrascrivere
    index=False
)

In [ ]:
cursor.execute("DROP TABLE IF EXISTS sell")

cursor.execute("""
CREATE TABLE sell (
    sale_id INTEGER PRIMARY KEY AUTOINCREMENT,
    barcode TEXT,
    espansione TEXT,
    nome TEXT,
    prezzo_stock REAL,
    prezzo_vendita REAL,
    sell_date TEXT
)
""")

conn.commit()

In [ ]:
conn.close()

In [ ]:

def sell_card(card_id):
    with sqlite3.connect("pokemon.db", timeout=10) as conn:
        cursor = conn.cursor()

        cursor.execute("""
            SELECT id, nome_completo, prezzo_eur, quantita_stock
            FROM stock
            WHERE id = ?
        """, (card_id,))

        card = cursor.fetchone()

        if card is None:
            return "Carta non trovata"

        print(card)
        id_, nome, prezzo, qty = card

        if qty <= 0:
            return "Nessun pezzo disponibile"

        sell_date = datetime.now().isoformat(sep=" ")

        # inserimento vendita (ORA OK)
        cursor.execute("""
            INSERT INTO sell (id, nome_completo, prezzo_eur, sell_date)
            VALUES (?, ?, ?, ?)
        """, (id_, nome, prezzo, sell_date))

        # update stock
        if qty >= 1:
            cursor.execute("""
                UPDATE stock
                SET quantita_stock = quantita_stock - 1
                WHERE id = ?
            """, (card_id,))
            print("Stock aggiornato")
        # else:
        #     cursor.execute("""
        #         DELETE FROM stock
        #         WHERE id = ?
        #     """, (card_id,))

In [ ]:
sell_card("BRS-184")